In [ ]:
# Problema: Construir un mart estrella para analizar la producción diaria de una fábrica por fecha y máquina.

import sqlite3
from pathlib import Path

import pandas as pd

ROOT = next(
    path
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (path / "data").is_dir() and (path / "submission").is_dir()
)
DATA = ROOT / "data"
MART = ROOT / "submission" / "factory_mart.db"


In [ ]:
# El hecho conserva el grano máquina-día; no se agrega antes de diseñar el mart.

throughput = pd.read_csv(DATA / "machine_throughput_export.csv")
uptime = pd.read_csv(DATA / "machine_uptime_export.csv")
facts = throughput.merge(
    uptime, on=["factory_id", "machine_id", "factory_date"], validate="one_to_one"
)
facts.shape


In [ ]:
# Las dimensiones describen las entidades por las que un analista filtrará o agrupará.

dim_factory = (
    facts[["factory_id"]]
    .drop_duplicates()
    .sort_values("factory_id")
    .reset_index(drop=True)
)
dim_factory["factory_key"] = dim_factory.index + 1
dim_machine = (
    facts[["factory_id", "machine_id"]]
    .drop_duplicates()
    .sort_values(["factory_id", "machine_id"])
    .reset_index(drop=True)
)
dim_machine["machine_key"] = dim_machine.index + 1
dim_date = (
    facts[["factory_date"]]
    .drop_duplicates()
    .sort_values("factory_date")
    .reset_index(drop=True)
)
dim_date["date_key"] = dim_date.index + 1
dim_date["date"] = pd.to_datetime(dim_date["factory_date"])
dim_date["year"] = dim_date.date.dt.year
dim_date["month"] = dim_date.date.dt.month


In [ ]:
# La tabla de hechos referencia dimensiones y guarda medidas aditivas o promediables.

fact_operations = (
    facts.merge(dim_factory, on="factory_id")
    .merge(dim_machine, on=["factory_id", "machine_id"])
    .merge(dim_date[["factory_date", "date_key"]], on="factory_date")[
        [
            "factory_key",
            "machine_key",
            "date_key",
            "daily_units_produced",
            "hours_operational",
        ]
    ]
)
fact_operations.shape


In [ ]:
MART.unlink(missing_ok=True)
with sqlite3.connect(MART) as connection:
    dim_factory[["factory_key", "factory_id"]].to_sql(
        "dim_factory", connection, index=False
    )
    dim_machine[["machine_key", "factory_id", "machine_id"]].to_sql(
        "dim_machine", connection, index=False
    )
    dim_date[["date_key", "factory_date", "year", "month"]].to_sql(
        "dim_date", connection, index=False
    )
    fact_operations.to_sql("fact_operations", connection, index=False)
    monthly = pd.read_sql_query(
        """
        SELECT d.year, d.month, f.factory_id, SUM(o.daily_units_produced) AS units_produced
        FROM fact_operations o JOIN dim_date d USING(date_key)
        JOIN dim_factory f USING(factory_key)
        GROUP BY d.year, d.month, f.factory_id
        ORDER BY d.year, d.month, f.factory_id
    """,
        connection,
    )
assert len(fact_operations) == len(facts)
monthly.head()
